In [0]:
from pyspark.sql import functions as F

# 1. leer los datos crudos desde la capa de bronze en Unity Catalog
df_bronze = spark.read.table("workspace.sch_oews_medallion.tb_brz_oews_2025")

# 2. Regla de Gobierno y Calidad (Data Quality Checks)
# La BLS usa '*', '#' o '**' cuando un dato salarial es confidencial o no disponible.
# Validamos que existan valores numéricos reales en el salario anual promedio (A_MEAN) y empleo (TOT_EMP).
# Exigimos datos numéricos válidos y filtramos solo nivel 'detailed' para evitar duplicidad por subtotales
regla_calidad = (
    (F.col("O_GROUP")=="detailed") &
    F.col("OCC_CODE").isNotNull() &
    F.col("AREA_TITLE").isNotNull() &
    (~F.col("A_MEAN").isin("*", "#", "**")) &
    (~F.col("TOT_EMP").isin("*", "#", "**")) &
    F.col("A_MEAN").isNotNull() &
    F.col("TOT_EMP").isNotNull()
)

# 3. Separar datos aprobados (Silver) de datos rechazados(Cuarentena)
df_silver_raw = df_bronze.filter(regla_calidad)
df_cuarentena = df_bronze.filter(~regla_calidad)

# 4. Estandarización y Tipado Estricto (Casting a números reales)
# Eliminamos posibles comas en los números y convertimos a formato numérico.
# Para crear nuevas columnas utilizamos una técnica de programación muy limpia llamada Encadenamiento de Métodos (Method Chaining).
df_silver_clean = (
    df_silver_raw
    .withColumn("tot_emp_num", F.regexp_replace(F.col("TOT_EMP"), ",", "").cast("int"))
    .withColumn("a_mean_num", F.regexp_replace(F.col("A_MEAN"), ",", "").cast("double"))
    .withColumn("h_mean_num", F.expr("try_cast(regexp_replace(H_MEAN, ',', '') as double)"))
    .withColumn("fecha_auditoria", F.current_timestamp()) # Trazabilidad de cuándo se gobernó el dato
)

# 5. Guardar ambas tablas en Unity Catalog
df_silver_clean.write.format("delta").mode("overwrite").saveAsTable("workspace.sch_oews_medallion.tb_slv_oews_clean")
df_cuarentena.write.format("delta").mode("overwrite").saveAsTable("workspace.sch_oews_medallion.tb_qua_oews_errors")


# 6. Imprimir reporte ejecutivo de Auditoría
total_inicial = df_bronze.count()
total_aprobados = df_silver_clean.count()
total_rechazados = df_cuarentena.count()
porcentaje_calidad = round((total_aprobados / total_inicial) * 100, 2)

print("=== REPORTE DE AUDITORÍA DE CALIDAD DE DATOS ===")
print(f"Total registros analizados (Bronze): {total_inicial}")
print(f"Registros aprobados (Silver): {total_aprobados} ({porcentaje_calidad}%)")
print(f"Registros enviados a Cuarentena: {total_rechazados}")


=== REPORTE DE AUDITORÍA DE CALIDAD DE DATOS ===
Total registros analizados (Bronze): 150023
Registros aprobados (Silver): 136276 (90.84%)
Registros enviados a Cuarentena: 13747


In [0]:
display(df_silver_clean.limit(10))

AREA,AREA_TITLE,AREA_TYPE,PRIM_STATE,NAICS,NAICS_TITLE,I_GROUP,OWN_CODE,OCC_CODE,OCC_TITLE,O_GROUP,TOT_EMP,EMP_PRSE,JOBS_1000,LOC_QUOTIENT,PCT_TOTAL,PCT_RPT,H_MEAN,A_MEAN,MEAN_PRSE,H_PCT10,H_PCT25,H_MEDIAN,H_PCT75,H_PCT90,A_PCT10,A_PCT25,A_MEDIAN,A_PCT75,A_PCT90,ANNUAL,HOURLY,tot_emp_num,a_mean_num,h_mean_num,fecha_auditoria
10180,"Abilene, TX",4,TX,000000,Cross-industry,cross-industry,1235,11-1011,Chief Executives,detailed,40,10.0,0.542,0.41,null,null,99.14,"206,200",6.3,42.50,58.65,79.70,113.51,187.68,"88,400","121,980","165,770","236,100","390,370",,,40,206200.0,99.14,2026-11-11T01:45:14.487Z
10180,"Abilene, TX",4,TX,000000,Cross-industry,cross-industry,1235,11-1021,General and Operations Managers,detailed,"2,120",1.6,28.241,1.25,null,null,54.63,"113,630",1.3,21.04,27.76,41.44,67.13,101.14,"43,770","57,740","86,200","139,630","210,360",,,2120,113630.0,54.63,2026-11-11T01:45:14.487Z
10180,"Abilene, TX",4,TX,000000,Cross-industry,cross-industry,1235,11-2021,Marketing Managers,detailed,140,6.5,1.848,0.73,null,null,61.98,"128,910",3.8,34.14,41.23,52.45,74.40,95.07,"71,010","85,770","109,100","154,750","197,740",,,140,128910.0,61.98,2026-11-11T01:45:14.487Z
10180,"Abilene, TX",4,TX,000000,Cross-industry,cross-industry,1235,11-2022,Sales Managers,detailed,280,7.1,3.694,0.90,null,null,61.86,"128,680",3.1,30.30,36.77,53.38,77.85,104.99,"63,020","76,470","111,020","161,920","218,380",,,280,128680.0,61.86,2026-11-11T01:45:14.487Z
10180,"Abilene, TX",4,TX,000000,Cross-industry,cross-industry,1235,11-2032,Public Relations Managers,detailed,40,14.1,0.530,1.10,null,null,49.70,"103,370",3.4,29.02,33.65,45.05,53.44,83.39,"60,370","69,990","93,710","111,160","173,440",,,40,103370.0,49.7,2026-11-11T01:45:14.487Z
10180,"Abilene, TX",4,TX,000000,Cross-industry,cross-industry,1235,11-3012,Administrative Services Managers,detailed,120,4.4,1.545,0.91,null,null,59.10,"122,930",4.8,34.03,39.94,49.86,64.90,80.66,"70,790","83,080","103,710","134,990","167,780",,,120,122930.0,59.1,2026-11-11T01:45:14.487Z
10180,"Abilene, TX",4,TX,000000,Cross-industry,cross-industry,1235,11-3013,Facilities Managers,detailed,70,14.1,0.902,0.90,null,null,41.24,"85,770",7.0,27.06,31.72,39.14,46.74,63.71,"56,290","65,990","81,410","97,220","132,520",,,70,85770.0,41.24,2026-11-11T01:45:14.487Z
10180,"Abilene, TX",4,TX,000000,Cross-industry,cross-industry,1235,11-3021,Computer and Information Systems Managers,detailed,120,9.3,1.653,0.38,null,null,67.10,"139,570",2.1,39.26,48.53,65.23,79.84,96.25,"81,660","100,940","135,670","166,070","200,200",,,120,139570.0,67.1,2026-11-11T01:45:14.487Z
10180,"Abilene, TX",4,TX,000000,Cross-industry,cross-industry,1235,11-3031,Financial Managers,detailed,310,8.5,4.193,0.77,null,null,76.61,"159,340",1.3,39.73,49.81,64.87,89.98,133.04,"82,640","103,600","134,920","187,160","276,710",,,310,159340.0,76.61,2026-11-11T01:45:14.487Z
10180,"Abilene, TX",4,TX,000000,Cross-industry,cross-industry,1235,11-3051,Industrial Production Managers,detailed,90,9.8,1.220,0.77,null,null,56.66,"117,860",2.5,34.91,39.86,52.32,65.24,85.79,"72,610","82,900","108,830","135,700","178,450",,,90,117860.0,56.66,2026-11-11T01:45:14.487Z
